In [ ]:
!pip install gensim

In [ ]:
# Install / Import Hugging Face Transformers
!pip install transformers

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder  # Label Encoding

from sklearn.model_selection import train_test_split  # Train Test Split

from sklearn.feature_extraction.text import TfidfVectorizer  # Feature Extraction (TF-IDF)

# Word2Vec (Word Embeddings)
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

# Bert Embeddings
from transformers import BertTokenizer, BertModel
import torch

# Classical ML Linear Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV


# Evaluation
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Data Plotting
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv('/content/reddit_cleaned_ml.csv')
df

In [ ]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df

| Index | Text                                                                                                   | Label       |
|-------:|--------------------------------------------------------------------------------------------------------|--------------|
| 0 | luvox causing me really bad night sweats so i’m... | OCD |
| 1 | anyone here still using push notifications for... | Normal |
| 2 | wanna do it sooooo bad too much of a pussy bec... | Suicidal |
| 3 | hahahah i just told the only ppl who still tal... | Depression |
| 4 | why is it that whenever i deliberately try to ... | ADHD |
| ... | ... | ... |
| 15908 | question for fellow veterans with ndas how do ... | PTSD |
| 15909 | i got a job! something positive to post i supp... | Anxiety |
| 15910 | my parents won’t test me for adhd for context ... | ADHD |
| 15911 | never sharing suicidal thoughts with anyone wh... | Suicidal |
| 15912 | feel like everything is going wrong, 19 and no... | Depression |

**Total Samples:** 15,913  
**Columns:** `text`, `label`

# Label Encoding

In [ ]:
#from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
df['label_encoded'] = encoder.fit_transform(df['label'])

# To see the mapping
label_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
for label, encoded in label_mapping.items():
    print(f"{label}: {encoded}")

| Label Name | Encoded Value |
|-------------|---------------|
| ADHD | 0 |
| Addiction | 1 |
| Anxiety | 2 |
| Depression | 3 |
| Normal | 4 |
| OCD | 5 |
| PTSD | 6 |
| Suicidal | 7 |

# Train Test Split

In [ ]:
#from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['label_encoded'],
    test_size=0.2,
    stratify=df['label'] ,
    random_state=42)


In [ ]:
print(X_train.shape)
print(X_test.shape)

## **Feature Extraction : TF-IDF vectorizer**

In [ ]:
# from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_df = 0.85,         # ignore (common)words that appear in more than 85% of posts
    min_df = 5,            # ignore words that appear in less than 5 posts
    max_features=10000,
    stop_words='english',
    ngram_range=(1, 3) # unigrams + trigrams
)

# Fit on training data and transform both train and test
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## **1. Baseline : Logistic Regression-(TF-IDF)**

In [ ]:
# from sklearn.linear_model import LogisticRegression

# initialize logistic regression model
lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    solver = 'liblinear',
    multi_class='auto'
)

# train the model
lr.fit(X_train_tfidf, y_train)

# predictions
y_pred_lr = lr.predict(X_test_tfidf)


In [ ]:
# from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'LogReg-Accuracy : {acc_lr}')

# Low precision = it often predicts positive incorrectly (many FP).
# Low recall = it misses many positives. (many FN)
# F1 score = harmonic mean of precision and recall
cr_lr = classification_report(y_test, y_pred_lr)


print("\nClassification Report (Logistic Regression-(TF-IDF)):\n", cr_lr)

#### **Logistic Regression + TF-IDF**
- **Accuracy:** 0.7713
- **Classification report :**
| Label       | Precision | Recall | F1-score | Support |
|------------|-----------|--------|----------|---------|
| 0          | 0.86      | 0.81   | 0.84     | 400     |
| 1          | 0.87      | 0.83   | 0.85     | 400     |
| 2          | 0.78      | 0.75   | 0.76     | 400     |
| 3          | 0.61      | 0.66   | 0.63     | 400     |
| 4          | 0.67      | 0.83   | 0.74     | 400     |
| 5          | 0.92      | 0.83   | 0.87     | 400     |
| 6          | 0.89      | 0.77   | 0.82     | 400     |
| 7          | 0.66      | 0.69   | 0.67     | 383     |

**Macro Avg:** P=0.78, R=0.77, F1=0.77  
**Weighted Avg:** P=0.78, R=0.77, F1=0.77

In [ ]:
def plot_confusion_matrix(y_test, y_pred, title):

  # Compute confusion matrix
  cm = confusion_matrix(y_test, y_pred)

  # decoded classes name
  class_names = encoder.classes_  # 'ADHD' : 0, 'Addiction' : 1  .....

  plt.figure(figsize=(10,7))
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
  plt.xlabel('Predicted')
  plt.ylabel('True')
  plt.title(title)
  plt.show()

In [ ]:
plot_confusion_matrix(y_test, y_pred_lr, title = "Confusion Matrix of Logistic Regression-(TF-IDF)")

**Git-hub-image-link :**
[Confusion Matrix of Logistic Regression-(TF-IDF)](https://github.com/TuliDas/MindScan-NLP/blob/main/results/cm-logreg-tfidf.png)

#### **Testing the LogisticRegression-TfIdf Model**

contribution=TF-IDF value of token×model coefficient for that class

We can extract the top supporting words (highest positive contributions) and top opposing words (most negative contributions).

In [ ]:

def predict_logReg(new_post, model, vectorizer, top_n = 6):

  # 1. Transform input using trained TF-IDF
  X_new = vectorizer.transform([new_post])     # tfidf score for each features/tokens

  # 2. Get probability distribution across classes
  probs = model.predict_proba(X_new)[0]

  # 3. Get Predicted class index
  pred_class_idx = probs.argmax()

  # 4. Map back to original label name
  pred_class = encoder.classes_[pred_class_idx]

  # 5. Get confidence score
  confidence = probs[pred_class_idx]

  # Get Token Contributions
  feature_names = vectorizer.get_feature_names_out()     # token names of current vectorizer
  coefs = model.coef_[pred_class_idx]                    # coefficients for predicted class
  X_array = X_new.toarray().ravel()                      # array of vectorizer's score

  contributions = [(feature_names[i], coefs[i]*X_array[i])
                   for i in np.where(X_array !=0)[0]]

  # Sort by contribution
  supporting = sorted(contributions, key=lambda x:x[1], reverse=True)[:top_n]


  # print results
  print("Input Post:", new_post)
  print(f"Predicted Class: {pred_class}")
  print("Confidence Score:", round(confidence,3))
  print("\n-----All class probabilities:-----")
  for label, p in zip(encoder.classes_, probs):
      print(f"{label}: {p:.3f}")

  print("\nTop supporting tokens:")
  for token, contrib in supporting:
        print(f"{token}: {round(contrib, 3)}")


In [ ]:
# Example 1 (Label = Depression)
input_post = "Lately I feel empty, nothing excites me anymore. Even simple things like eating or talking to friends feel like a burden. I just want to sleep all day and avoid facing the world."

predict_logReg(input_post, lr, tfidf)


**Input Post:**  
> Lately I feel empty, nothing excites me anymore. Even simple things like eating or talking to friends feel like a burden. I just want to sleep all day and avoid facing the world.

**Predicted Class:** `Depression`  
**Confidence Score:** `0.494`

---

#### All Class Probabilities

| Class | Probability |
|:------|:-------------|
| ADHD | 0.040 |
| Addiction | 0.043 |
| Anxiety | 0.064 |
| **Depression** | **0.494** |
| Normal | 0.054 |
| OCD | 0.084 |
| PTSD | 0.147 |
| Suicidal | 0.073 |

---

#### Top Supporting Tokens

| Token | Weight |
|:------|:--------|
| feel | 0.550 |
| anymore | 0.518 |
| lately | 0.277 |
| want | 0.238 |
| burden | 0.203 |
| sleep | 0.126 |


In [ ]:
# Example 2 (Label = PTSD ; src : Huggingface)
input_post = " I had some violent flashbacks and brutal compulsive images last night and I know I shouldn’t cut but it’s the only thing that gives me some sense of control right now. I’m losing everyone and everything because of what happened to me when I was younger and what I continue to allowed to happen to me as I got older. I’ve dumped all my trauma on the only person who has loved me unconditionally and now I may lose him too. Fuck what happened to me."

predict_logReg(input_post, lr, tfidf)


**Input Post:**  
> I had some violent flashbacks and brutal compulsive images last night and I know I shouldn’t cut but it’s the only thing that gives me some sense of control right now. I’m losing everyone and everything because of what happened to me when I was younger and what I continue to allowed to happen to me as I got older. I’ve dumped all my trauma on the only person who has loved me unconditionally and now I may lose him too. Fuck what happened to me.

**Predicted Class:** `PTSD`  
**Confidence Score:** `0.498`

---

#### All Class Probabilities

| Class | Probability |
|:------|:-------------|
| ADHD | 0.037 |
| Addiction | 0.062 |
| Anxiety | 0.034 |
| Depression | 0.106 |
| Normal | 0.024 |
| OCD | 0.057 |
| **PTSD** | **0.498** |
| Suicidal | 0.182 |

---

####Top Supporting Tokens

| Token | Weight |
|:------|:--------|
| trauma | 1.743 |
| happen | 0.419 |
| control | 0.155 |
| violent | 0.148 |
| night | 0.146 |
| sense | 0.058 |

In [ ]:
# Example 3 (Label = Suicidal)
input_post = "I dont think my life is worthy anymore. Failure, heartbreak , parent's death have destroyed me completely. why would I live ? maybe today is my last day, and its my last post ... have a good life fellas"

predict_logReg(input_post, lr, tfidf)

**Input Post:**  
> I dont think my life is worthy anymore. Failure, heartbreak, parent's death have destroyed me completely. Why would I live? Maybe today is my last day, and it's my last post ... have a good life fellas.

**Predicted Class:** `Suicidal`  
**Confidence Score:** `0.528`

---

#### All Class Probabilities

| Class | Probability |
|:------|:-------------|
| ADHD | 0.013 |
| Addiction | 0.103 |
| Anxiety | 0.020 |
| Depression | 0.266 |
| Normal | 0.037 |
| OCD | 0.015 |
| PTSD | 0.019 |
| **Suicidal** | **0.528** |

---

#### Top Supporting Tokens

| Token | Weight |
|:------|:--------|
| life | 0.855 |
| death | 0.795 |
| live | 0.382 |
| anymore | 0.237 |
| parent | 0.221 |
| think life | 0.184 |

In [ ]:
# Example 4 (Label : Depression)
input_post = "I don’t know what I will do in the future. I feel like I’ve wasted three years of my life. That relationship was a curse. Most of my friends have almost completed their PhDs, and here I am, just working on simple ML and NLP projects. Sometimes I wonder how things turned out this way. I never thought I would struggle so much with my academic life, especially since I was always the top girl in school and college. Life has changed so drastically. I don’t even know how I can ever catch up with my friends in the academic field."

predict_logReg(input_post, lr, tfidf)

**Input Post:**  
> I don’t know what I will do in the future. I feel like I’ve wasted three years of my life. That relationship was a curse. Most of my friends have almost completed their PhDs, and here I am, just working on simple ML and NLP projects. Sometimes I wonder how things turned out this way. I never thought I would struggle so much with my academic life, especially since I was always the top girl in school and college. Life has changed so drastically. I don’t even know how I can ever catch up with my friends in the academic field.

**Predicted Class:** `Depression`  
**Confidence Score:** `0.297`

---

#### All Class Probabilities

| Class | Probability |
|:------|:-------------|
| ADHD | 0.113 |
| Addiction | 0.045 |
| Anxiety | 0.099 |
| **Depression** | **0.297** |
| Normal | 0.133 |
| OCD | 0.124 |
| PTSD | 0.059 |
| Suicidal | 0.129 |

---

#### Top Supporting Tokens

| Token | Weight |
|:------|:--------|
| life | 0.394 |
| know | 0.192 |
| feel | 0.174 |
| college | 0.165 |
| relationship | 0.134 |
| school | 0.110 |


In [ ]:
# Example 5 (Label = Anxiety)
input_post = "I keep replaying small conversations in my head, wondering if I sounded rude or stupid 🤦‍♀️. Hours later my chest feels tight 💔 just remembering how I said ‘hi’ wrong or didn’t smile enough. People probably don’t even notice, but my mind won’t stop. I cancel plans often because it feels safer to avoid messing up again 😞."

predict_logReg(input_post, lr, tfidf)

**Input Post:**  
> I keep replaying small conversations in my head, wondering if I sounded rude or stupid 🤦‍♀️. Hours later my chest feels tight 💔 just remembering how I said ‘hi’ wrong or didn’t smile enough. People probably don’t even notice, but my mind won’t stop. I cancel plans often because it feels safer to avoid messing up again 😞.

**Predicted Class:** `Anxiety`  
**Confidence Score:** `0.344`

---

#### All Class Probabilities

| Class | Probability |
|:------|:-------------|
| ADHD | 0.047 |
| Addiction | 0.061 |
| **Anxiety** | **0.344** |
| Depression | 0.066 |
| Normal | 0.241 |
| OCD | 0.167 |
| PTSD | 0.042 |
| Suicidal | 0.032 |

---

#### Top Supporting Tokens

| Token | Weight |
|:------|:--------|
| chest | 0.447 |
| people | 0.276 |
| probably | 0.226 |
| avoid | 0.138 |
| wrong | 0.128 |
| rude | 0.126 |

## **2. Classical ML : SVM + TF-IDF**

In [ ]:
#from sklearn.svm import LinearSVC
#from sklearn.calibration import CalibratedClassifierCV

# Train SVM Base for Token Contribution
svm_base = LinearSVC(class_weight='balanced', random_state=42)
svm_base.fit(X_train_tfidf, y_train)

# 2. Train calibrated SVM for Predict and probability scores
svm_cal = CalibratedClassifierCV(svm_base, cv=5)
svm_cal.fit(X_train_tfidf, y_train)

y_pred_svm = svm_cal.predict(X_test_tfidf)

In [ ]:
acc_svm = accuracy_score(y_test, y_pred_svm)
cr_svm = classification_report(y_test, y_pred_svm)
print(f"SVM-Accuracy : {acc_svm}")
print(f"Classification Report (SVM): \n{cr_svm}")

## **2. SVM + TF-IDF**
**Accuracy:** 0.7716

| Label       | Precision | Recall | F1-score | Support |
|------------|-----------|--------|----------|---------|
| 0          | 0.83      | 0.83   | 0.83     | 400     |
| 1          | 0.86      | 0.86   | 0.86     | 400     |
| 2          | 0.76      | 0.77   | 0.76     | 400     |
| 3          | 0.60      | 0.64   | 0.62     | 400     |
| 4          | 0.72      | 0.78   | 0.75     | 400     |
| 5          | 0.92      | 0.84   | 0.88     | 400     |
| 6          | 0.85      | 0.78   | 0.81     | 400     |
| 7          | 0.66      | 0.66   | 0.66     | 383     |

**Macro Avg:** P=0.78, R=0.77, F1=0.77  
**Weighted Avg:** P=0.78, R=0.77, F1=0.77


In [ ]:
# Plot Confusion Matrix (SVM)

# Compute confusion matrix
plot_confusion_matrix(y_test, y_pred_svm, title = "Confusion Matrix of SVM (TF-IDF)")

**Git-Hub-Image-Link :** [Confusion Matrix of SVM (TF-IDF)](https://github.com/TuliDas/MindScan-NLP/blob/main/results/cm-svm-tfidf.png)

In [ ]:
def predict_svm(new_post, model_base, model_cal, vectorizer, top_n = 6):

  # 1. Transform input using trained TF-IDF
  X_new = vectorizer.transform([new_post])     # tfidf score for each features/tokens

  # 2. Get probability distribution across classes
  probs = model_cal.predict_proba(X_new)[0]

  # 3. Get Predicted class index
  pred_class_idx = probs.argmax()

  # 4. Map back to original label name
  pred_class = encoder.classes_[pred_class_idx]

  # 5. Get confidence score
  confidence = probs[pred_class_idx]

  # Get Token Contributions
  feature_names = vectorizer.get_feature_names_out()     # token names of current vectorizer
  coefs = model_base.coef_[pred_class_idx]     # coefficients for predicted class (SVM model)
  X_array = X_new.toarray().ravel()                      # array of vectorizer's score

  contributions = [(feature_names[i], coefs[i]*X_array[i])
                   for i in np.where(X_array !=0)[0]]

  # Sort by contribution
  supporting = sorted(contributions, key=lambda x:x[1], reverse=True)[:top_n]


  # print results
  print("Input Post:", new_post)
  print(f"Predicted Class: {pred_class}")
  print("Confidence Score:", round(confidence,3))
  print("\n-----All class probabilities:-----")
  for label, p in zip(encoder.classes_, probs):
      print(f"{label}: {p:.3f}")

  print("\nTop supporting tokens:")
  for token, contrib in supporting:
        print(f"{token}: {round(contrib, 3)}")

In [ ]:
# Example 1 (Label = Depression ; src = ChatGPT)
input_post = "Lately I feel empty, nothing excites me anymore. Even simple things like eating or talking to friends feel like a burden. I just want to sleep all day and avoid facing the world."

predict_svm(input_post, svm_base, svm_cal, tfidf)

**Input Post:**  
> Lately I feel empty, nothing excites me anymore. Even simple things like eating or talking to friends feel like a burden. I just want to sleep all day and avoid facing the world.

**Predicted Class:** `Depression`  
**Confidence Score:** `0.615`

---

#### All Class Probabilities

| Class | Probability |
|:------|:-------------|
| ADHD | 0.006 |
| Addiction | 0.015 |
| Anxiety | 0.028 |
| **Depression** | **0.615** |
| Normal | 0.054 |
| OCD | 0.085 |
| PTSD | 0.168 |
| Suicidal | 0.028 |

---

#### Top Supporting Tokens

| Token | Weight |
|:------|:--------|
| lately | 0.258 |
| feel | 0.236 |
| anymore | 0.212 |
| burden | 0.142 |
| eating | 0.135 |
| sleep day | 0.103 |

In [ ]:
# Example 2 (Label = PTSD ; src : Huggingface)
input_post = " I had some violent flashbacks and brutal compulsive images last night and I know I shouldn’t cut but it’s the only thing that gives me some sense of control right now. I’m losing everyone and everything because of what happened to me when I was younger and what I continue to allowed to happen to me as I got older. I’ve dumped all my trauma on the only person who has loved me unconditionally and now I may lose him too. Fuck what happened to me."

predict_svm(input_post, svm_base, svm_cal, tfidf)

**Input Post:**  
> I had some violent flashbacks and brutal compulsive images last night and I know I shouldn’t cut but it’s the only thing that gives me some sense of control right now. I’m losing everyone and everything because of what happened to me when I was younger and what I continue to allowed to happen to me as I got older. I’ve dumped all my trauma on the only person who has loved me unconditionally and now I may lose him too. Fuck what happened to me.

**Predicted Class:** `PTSD`  
**Confidence Score:** `0.629`

---

#### All Class Probabilities

| Class | Probability |
|:------|:-------------|
| ADHD | 0.005 |
| Addiction | 0.043 |
| Anxiety | 0.015 |
| Depression | 0.116 |
| Normal | 0.001 |
| OCD | 0.009 |
| **PTSD** | **0.629** |
| Suicidal | 0.182 |

---

#### Top Supporting Tokens

| Token | Weight |
|:------|:--------|
| trauma | 0.901 |
| happen | 0.163 |
| control | 0.108 |
| night | 0.080 |
| thing | 0.045 |
| night know | 0.045 |

In [ ]:
# Example 3 (Label = Suicidal)
input_post = "I dont think my life is worthy anymore. Failure, heartbreak , parent's death have destroyed me completely. why would I live ? maybe today is my last day, and its my last post ... have a good life fellas"

predict_svm(input_post, svm_base, svm_cal, tfidf)

In [ ]:
# Example 4 (Label : Depression)
input_post = "I don’t know what I will do in the future. I feel like I’ve wasted three years of my life. That relationship was a curse. Most of my friends have almost completed their PhDs, and here I am, just working on simple ML and NLP projects. Sometimes I wonder how things turned out this way. I never thought I would struggle so much with my academic life, especially since I was always the top girl in school and college. Life has changed so drastically. I don’t even know how I can ever catch up with my friends in the academic field."

predict_svm(input_post, svm_base, svm_cal, tfidf)

**Input Post:**  
> I don’t know what I will do in the future. I feel like I’ve wasted three years of my life. That relationship was a curse. Most of my friends have almost completed their PhDs, and here I am, just working on simple ML and NLP projects. Sometimes I wonder how things turned out this way. I never thought I would struggle so much with my academic life, especially since I was always the top girl in school and college. Life has changed so drastically. I don’t even know how I can ever catch up with my friends in the academic field.

**Predicted Class:** `Depression`  
**Confidence Score:** `0.276`

---

#### All Class Probabilities

| Class | Probability |
|:------|:-------------|
| ADHD | 0.169 |
| Addiction | 0.031 |
| Anxiety | 0.167 |
| **Depression** | **0.276** |
| Normal | 0.130 |
| OCD | 0.105 |
| PTSD | 0.014 |
| Suicidal | 0.109 |

---

#### Top Supporting Tokens

| Token | Weight |
|:------|:--------|
| life | 0.124 |
| relationship | 0.077 |
| feel | 0.075 |
| school | 0.062 |
| girl | 0.050 |
| know | 0.048 |

In [ ]:
# Example 5 (Label = Anxiety)
input_post = "I keep replaying small conversations in my head, wondering if I sounded rude or stupid 🤦‍♀️. Hours later my chest feels tight 💔 just remembering how I said ‘hi’ wrong or didn’t smile enough. People probably don’t even notice, but my mind won’t stop. I cancel plans often because it feels safer to avoid messing up again 😞."

predict_svm(input_post, svm_base, svm_cal, tfidf)

**Input Post:**  
> I keep replaying small conversations in my head, wondering if I sounded rude or stupid 🤦‍♀️. Hours later my chest feels tight 💔 just remembering how I said ‘hi’ wrong or didn’t smile enough. People probably don’t even notice, but my mind won’t stop. I cancel plans often because it feels safer to avoid messing up again 😞.

**Predicted Class:** `Anxiety`  
**Confidence Score:** `0.394`

---

#### All Class Probabilities

| Class | Probability |
|:------|:-------------|
| ADHD | 0.010 |
| Addiction | 0.039 |
| **Anxiety** | **0.394** |
| Depression | 0.065 |
| Normal | 0.262 |
| OCD | 0.218 |
| PTSD | 0.005 |
| Suicidal | 0.007 |

---

#### Top Supporting Tokens

| Token | Weight |
|:------|:--------|
| chest | 0.256 |
| rude | 0.141 |
| probably | 0.136 |
| people | 0.098 |
| avoid | 0.096 |
| smile | 0.076 |

## **Classical ML: (with embeddings)**
- Logistic Regression with Word2Vec embeddings
- Logistic Regression with Bert Embeddings

In [ ]:
# -------- Word2Vec -------------

#from gensim.models import Word2Vec
#from gensim.utils import simple_preprocess

# 1. Prepare corpus (tokenized texts)
corpus = [text.split() for text in df['text']]  # Simple Tokenization

# 2. Train Word2Vec
w2v_model = Word2Vec(sentences=corpus,
                     vector_size=100,
                     window=5,
                     min_count=2,
                     workers=4 )

In [ ]:
# 3. Convert each word to average Word2Vec embedding

def sentence_vector(sentence, model, vector_size=100):
    words = sentence.split()
    vecs = [model.wv[word] for word in words if word in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(vector_size)


train_texts = df.loc[X_train.index, 'text']
test_texts = df.loc[X_test.index, 'text']
train_label = df.loc[y_train.index, 'label_encoded']
test_label = df.loc[y_test.index, 'label_encoded']

# 4. Generate Word2Vec for train test set
X_train_w2v = [sentence_vector(t,w2v_model) for t in train_texts]
X_test_w2v = [sentence_vector(t,w2v_model) for t in test_texts]

In [ ]:
# 5. new LR model for embeddings
lr_w2v = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

lr_w2v.fit(X_train_w2v, y_train)
y_pred_lr_w2v = lr_w2v.predict(X_test_w2v)

In [ ]:
acc_lr_w2v = accuracy_score(y_test, y_pred_lr_w2v)
print(f'LogReg-Accuracy : {acc_lr_w2v}')

cr_lr_w2v = classification_report(y_test, y_pred_lr_w2v)
print("Classification Reports of LogReg/Word2Vec : ")
print(cr_lr_w2v)

## **3. Logistic Regression + Word2Vec**
**Accuracy:** 0.6591

| Label       | Precision | Recall | F1-score | Support |
|------------|-----------|--------|----------|---------|
| 0          | 0.70      | 0.70   | 0.70     | 400     |
| 1          | 0.78      | 0.72   | 0.75     | 400     |
| 2          | 0.69      | 0.63   | 0.66     | 400     |
| 3          | 0.48      | 0.46   | 0.47     | 400     |
| 4          | 0.60      | 0.70   | 0.65     | 400     |
| 5          | 0.78      | 0.73   | 0.75     | 400     |
| 6          | 0.74      | 0.73   | 0.73     | 400     |
| 7          | 0.55      | 0.60   | 0.57     | 383     |

**Macro Avg:** P=0.66, R=0.66, F1=0.66  
**Weighted Avg:** P=0.66, R=0.66, F1=0.66

In [ ]:
plot_confusion_matrix(y_test, y_pred_lr_w2v, title="Confusion Matrix of LogReg / word2vec embeddings")

**GitHub-Image-Link :**[Confusion Matrix of LogReg / word2vec embeddings](https://github.com/TuliDas/MindScan-NLP/blob/main/results/cm_logreg-w2v.png)

In [ ]:
def predict_logreg_w2v(new_post, model, w2v_model):
  X_new = [sentence_vector(new_post, w2v_model)]

  # Predict
  probs = model.predict_proba(X_new)[0]
  pred_class_idx = probs.argmax()
  pred_class = encoder.classes_[pred_class_idx]
  confidence = probs[pred_class_idx]

  # print results
  print("\nInput Post:", new_post)
  print(f"Predicted Class: {pred_class}")
  print("Confidence Score:", round(confidence,3))
  print("\n-----All class probabilities:-----")
  for label, p in zip(encoder.classes_, probs):
      print(f"{label}: {p:.3f}")


In [ ]:
# Example 1 (Label = Depression ; src = ChatGPT)
input_post = "Lately I feel empty, nothing excites me anymore. Even simple things like eating or talking to friends feel like a burden. I just want to sleep all day and avoid facing the world."
predict_logreg_w2v(input_post, lr_w2v, w2v_model)

**Input Post :**  
> Lately I feel empty, nothing excites me anymore. Even simple things like eating or talking to friends feel like a burden. I just want to sleep all day and avoid facing the world.


**Predicted Class:** `Depression`  
**Confidence Score:** `0.325`

**All class probabilities:**
| Class | Probability |
|--------|--------------|
| ADHD | 0.119 |
| Addiction | 0.119 |
| Anxiety | 0.073 |
| **Depression** | **0.325** |
| Normal | 0.094 |
| OCD | 0.079 |
| PTSD | 0.125 |
| Suicidal | 0.067 |

In [ ]:
# Example 2 (Label = PTSD ; src : Huggingface)
input_post = " I had some violent flashbacks and brutal compulsive images last night and I know I shouldn’t cut but it’s the only thing that gives me some sense of control right now. I’m losing everyone and everything because of what happened to me when I was younger and what I continue to allowed to happen to me as I got older. I’ve dumped all my trauma on the only person who has loved me unconditionally and now I may lose him too. Fuck what happened to me."
predict_logreg_w2v(input_post, lr_w2v, w2v_model)

**Input Post:**  
> I had some violent flashbacks and brutal compulsive images last night and I know I shouldn’t cut but it’s the only thing that gives me some sense of control right now. I’m losing everyone and everything because of what happened to me when I was younger and what I continue to allowed to happen to me as I got older. I’ve dumped all my trauma on the only person who has loved me unconditionally and now I may lose him too. Fuck what happened to me.

**Predicted Class:** `PTSD`  
**Confidence Score:** `0.568`

---

**All class probabilities:**
| Class | Probability |
|--------|--------------|
| ADHD | 0.036 |
| Addiction | 0.020 |
| Anxiety | 0.040 |
| Depression | 0.045 |
| Normal | 0.095 |
| OCD | 0.141 |
| **PTSD** | **0.568** |
| Suicidal | 0.055 |


In [ ]:
# Example 3 (Label = Suicidal)
input_post = "I dont think my life is worthy anymore. Failure, heartbreak , parent's death have destroyed me completely. why would I live ? maybe today is my last day, and its my last post ... have a good life fellas"

predict_logreg_w2v(input_post, lr_w2v, w2v_model)

**Input Post :**  
> I dont think my life is worthy anymore. Failure, heartbreak, parent's death have destroyed me completely. Why would I live? Maybe today is my last day, and it's my last post ... have a good life fellas.


**Predicted Class:** `Suicidal`  
**Confidence Score:** `0.586`

**All class probabilities:**
| Class | Probability |
|--------|--------------|
| ADHD | 0.003 |
| Addiction | 0.146 |
| Anxiety | 0.033 |
| Depression | 0.149 |
| Normal | 0.047 |
| OCD | 0.017 |
| PTSD | 0.019 |
| **Suicidal** | **0.586** |


In [ ]:
# Example 4 (Label = Depression)
input_post = "I don’t know what I will do in the future. I feel like I’ve wasted three years of my life. That relationship was a curse. Most of my friends have almost completed their PhDs, and here I am, just working on simple ML and NLP projects. Sometimes I wonder how things turned out this way. I never thought I would struggle so much with my academic life, especially since I was always the top girl in school and college. Life has changed so drastically. I don’t even know how I can ever catch up with my friends in the academic field."

predict_logreg_w2v(input_post, lr_w2v, w2v_model)

**Input Post :**  
> I don’t know what I will do in the future. I feel like I’ve wasted three years of my life. That relationship was a curse. Most of my friends have almost completed their PhDs, and here I am, just working on simple ML and NLP projects. Sometimes I wonder how things turned out this way. I never thought I would struggle so much with my academic life, especially since I was always the top girl in school and college. Life has changed so drastically. I don’t even know how I can ever catch up with my friends in the academic field.

**Predicted Class:** `ADHD`  
**Confidence Score:** `0.249`

**All class probabilities:**
| Class | Probability |
|--------|--------------|
| **ADHD** | **0.249** |
| Addiction | 0.041 |
| Anxiety | 0.059 |
| Depression | 0.126 |
| Normal | 0.246 |
| OCD | 0.120 |
| PTSD | 0.123 |
| Suicidal | 0.035 |


In [ ]:
# Example 5 (Label = Anxiety)
input_post = "I keep replaying small conversations in my head, wondering if I sounded rude or stupid 🤦‍♀️. Hours later my chest feels tight 💔 just remembering how I said ‘hi’ wrong or didn’t smile enough. People probably don’t even notice, but my mind won’t stop. I cancel plans often because it feels safer to avoid messing up again 😞."

predict_logreg_w2v(input_post, lr_w2v, w2v_model)

**Input Post :**  
> I keep replaying small conversations in my head, wondering if I sounded rude or stupid 🤦‍♀️. Hours later my chest feels tight 💔 just remembering how I said ‘hi’ wrong or didn’t smile enough. People probably don’t even notice, but my mind won’t stop. I cancel plans often because it feels safer to avoid messing up again 😞.

**Predicted Class:** `Normal`  
**Confidence Score:** `0.418`

**All class probabilities:**
| Class | Probability |
|--------|--------------|
| ADHD | 0.067 |
| Addiction | 0.030 |
| Anxiety | 0.203 |
| Depression | 0.025 |
| **Normal** | **0.418** |
| OCD | 0.166 |
| PTSD | 0.032 |
| Suicidal | 0.059 |

In [ ]:
# ------------ Bert Embeddings -------------


In [ ]:
# 1. Load base uncased BERT
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased")

In [ ]:
from tqdm import tqdm

def get_bert_embeddings(texts, tokenizer, model, max_len=128, batch_size=32, device="cuda"):
    model.to(device)
    model.eval()
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]

        # Tokenize in batch
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        # Mean pooling
        mean_embeds = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        all_embeddings.extend(mean_embeds)

    return np.array(all_embeddings)

In [ ]:
# 3. Generate Embeddings for Train/Test
X_train_bert = get_bert_embeddings(X_train, tokenizer, bert_model, batch_size=32, device="cuda" if torch.cuda.is_available() else "cpu")
X_test_bert  = get_bert_embeddings(X_test, tokenizer, bert_model, batch_size=32, device="cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# 4. Train Logistic Regression on BERT embeddings

lr_bert = LogisticRegression(max_iter=2000, class_weight="balanced")
lr_bert.fit(X_train_bert, y_train)

y_pred_bert = lr_bert.predict(X_test_bert)


In [ ]:
acc_lr_bert = accuracy_score(y_test, y_pred_bert)
print(f'LogReg-BERT-Accuracy : {acc_lr_bert}')
cr_lr_bert = classification_report(y_test, y_pred_bert)
print("Classification Report LoReg / (BERT Embeddings):\n", )
print(cr_lr_bert)

## **4. Logistic Regression + BERT Embeddings**
**Accuracy:** 0.6846

| Label       | Precision | Recall | F1-score | Support |
|------------|-----------|--------|----------|---------|
| 0          | 0.75      | 0.74   | 0.74     | 400     |
| 1          | 0.79      | 0.78   | 0.78     | 400     |
| 2          | 0.66      | 0.60   | 0.63     | 400     |
| 3          | 0.51      | 0.54   | 0.52     | 400     |
| 4          | 0.68      | 0.76   | 0.72     | 400     |
| 5          | 0.75      | 0.74   | 0.75     | 400     |
| 6          | 0.73      | 0.72   | 0.72     | 400     |
| 7          | 0.61      | 0.59   | 0.60     | 383     |

**Macro Avg:** P=0.68, R=0.68, F1=0.68  
**Weighted Avg:** P=0.69, R=0.68, F1=0.68

In [ ]:
plot_confusion_matrix(y_test, y_pred_bert, title="Confusion Matrix of LogReg / BERT embeddings")

**Github-Image-Link :**[Confusion Matrix of LogReg / BERT embeddings](https://github.com/TuliDas/MindScan-NLP/blob/main/results/cm-logreg-bert.png)



In [ ]:
import joblib

# Save Logistic Regression model
joblib.dump(lr_bert, "logreg_bert_model.pkl")

# Save label encoder
joblib.dump(encoder, "label_encoder.pkl")

# Save BERT embeddings (optional, usually better to recompute when predicting)
joblib.dump(X_train_bert, "X_train_bert.pkl")
joblib.dump(X_test_bert, "X_test_bert.pkl")


In [ ]:
def predict_logReg_bert(new_post, model):
  X_new = get_bert_embeddings([new_post], tokenizer, bert_model, batch_size=32, device="cuda" if torch.cuda.is_available() else "cpu")

  # Predict
  probs = model.predict_proba(X_new)[0]
  pred_class_idx = probs.argmax()
  pred_class = encoder.classes_[pred_class_idx]
  confidence = probs[pred_class_idx]

  # print results
  print("\nInput Post:", new_post)
  print(f"Predicted Class: {pred_class}")
  print("Confidence Score:", round(confidence,3))
  print("\n-----All class probabilities:-----")
  for label, p in zip(encoder.classes_, probs):
      print(f"{label}: {p:.3f}")


In [ ]:
# Example 1 (Label = Depression ; src = ChatGPT)
input_post = "Lately I feel empty, nothing excites me anymore. Even simple things like eating or talking to friends feel like a burden. I just want to sleep all day and avoid facing the world."

predict_logReg_bert(input_post, lr_bert)


**Input Post :**  
> Lately I feel empty, nothing excites me anymore. Even simple things like eating or talking to friends feel like a burden. I just want to sleep all day and avoid facing the world.

**Predicted Class:** `Depression`  
**Confidence Score:** `0.996`

**All class probabilities:**
| Class | Probability |
|--------|--------------|
| ADHD | 0.000 |
| Addiction | 0.000 |
| Anxiety | 0.001 |
| **Depression** | **0.996** |
| Normal | 0.001 |
| OCD | 0.000 |
| PTSD | 0.000 |
| Suicidal | 0.003 |


In [ ]:
# Example 2 (Label = PTSD ; src : Huggingface)
input_post = " I had some violent flashbacks and brutal compulsive images last night and I know I shouldn’t cut but it’s the only thing that gives me some sense of control right now. I’m losing everyone and everything because of what happened to me when I was younger and what I continue to allowed to happen to me as I got older. I’ve dumped all my trauma on the only person who has loved me unconditionally and now I may lose him too. Fuck what happened to me."


predict_logReg_bert(input_post, lr_bert)

**Input Post :**  
> I had some violent flashbacks and brutal compulsive images last night and I know I shouldn’t cut but it’s the only thing that gives me some sense of control right now. I’m losing everyone and everything because of what happened to me when I was younger and what I continue to allowed to happen to me as I got older. I’ve dumped all my trauma on the only person who has loved me unconditionally and now I may lose him too. Fuck what happened to me.

**Predicted Class:** `Suicidal`  
**Confidence Score:** `0.975`

**All class probabilities:**
| Class | Probability |
|--------|--------------|
| ADHD | 0.000 |
| Addiction | 0.000 |
| Anxiety | 0.000 |
| Depression | 0.008 |
| Normal | 0.000 |
| OCD | 0.000 |
| PTSD | 0.017 |
| **Suicidal** | **0.975** |


In [ ]:
# Example 3 (Label = Suicidal)
input_post = "I dont think my life is worthy anymore. Failure, heartbreak , parent's death have destroyed me completely. why would I live ? maybe today is my last day, and its my last post ... have a good life fellas"

predict_logReg_bert(input_post, lr_bert)

**Input Post :**  
> I dont think my life is worthy anymore. Failure, heartbreak , parent's death have destroyed me completely. why would I live ? maybe today is my last day, and its my last post ... have a good life fellas

**Predicted Class:** `Suicidal`  
**Confidence Score:** `0.965`

**All class probabilities:**
| Class | Probability |
|--------|--------------|
| ADHD | 0.000 |
| Addiction | 0.000 |
| Anxiety | 0.000 |
| Depression | 0.035 |
| Normal | 0.000 |
| OCD | 0.000 |
| PTSD | 0.000 |
| **Suicidal** | **0.965** |


In [ ]:
# Example 4 (Label = Depression)
input_post = "I don’t know what I will do in the future. I feel like I’ve wasted three years of my life. That relationship was a curse. Most of my friends have almost completed their PhDs, and here I am, just working on simple ML and NLP projects. Sometimes I wonder how things turned out this way. I never thought I would struggle so much with my academic life, especially since I was always the top girl in school and college. Life has changed so drastically. I don’t even know how I can ever catch up with my friends in the academic field."

predict_logReg_bert(input_post, lr_bert)

**Input Post :**  
> I don’t know what I will do in the future. I feel like I’ve wasted three years of my life. That relationship was a curse. Most of my friends have almost completed their PhDs, and here I am, just working on simple ML and NLP projects. Sometimes I wonder how things turned out this way. I never thought I would struggle so much with my academic life, especially since I was always the top girl in school and college. Life has changed so drastically. I don’t even know how I can ever catch up with my friends in the academic field.

**Predicted Class:** `Suicidal`  
**Confidence Score:** `0.836`

**All class probabilities:**
| Class | Probability |
|--------|--------------|
| ADHD | 0.000 |
| Addiction | 0.000 |
| Anxiety | 0.007 |
| Depression | 0.125 |
| Normal | 0.020 |
| OCD | 0.002 |
| PTSD | 0.010 |
| **Suicidal** | **0.836** |


In [ ]:
# Example 5 (Label = Anxiety)
input_post = "I keep replaying small conversations in my head, wondering if I sounded rude or stupid 🤦‍♀️. Hours later my chest feels tight 💔 just remembering how I said ‘hi’ wrong or didn’t smile enough. People probably don’t even notice, but my mind won’t stop. I cancel plans often because it feels safer to avoid messing up again 😞."

predict_logReg_bert(input_post, lr_bert)

**Input Post :**  
> I keep replaying small conversations in my head, wondering if I sounded rude or stupid 🤦‍♀️. Hours later my chest feels tight 💔 just remembering how I said ‘hi’ wrong or didn’t smile enough. People probably don’t even notice, but my mind won’t stop. I cancel plans often because it feels safer to avoid messing up again 😞.

**Predicted Class:** `Depression`  
**Confidence Score:** `0.405`

**All class probabilities:**
| Class | Probability |
|--------|--------------|
| ADHD | 0.000 |
| Addiction | 0.000 |
| Anxiety | 0.022 |
| **Depression** | **0.405** |
| Normal | 0.009 |
| OCD | 0.000 |
| PTSD | 0.349 |
| Suicidal | 0.215 |
